In [3]:
import pandas as pd
import numpy as np
#from sklearn import train_test_split

# select significantly reversed miRNA pairs
f = open('kazuki_data.json')
kazuki_data = pd.read_json(f, orient='split')
f.close()
f = open('watara_data.json')
watara_data = pd.read_json(f, orient='split')
f.close()
f = open('yokoi_data.json')
yokoi_data = pd.read_json(f, orient='split')
f.close()

#we need to have training/validation. 
#the article has a training split of the kazuki + yokoi (only 2000 noncancer and 200 OV)
# so im skipping test, so validation the rest of yokoi (759 noncancer,120 OV) and watara (100 noncancer, 40 OV)
#for training, I'm only going to use 'stable mirna pairs' as described in article, and need to maintain REO of like 95% or sum

training = pd.concat([kazuki_data, yokoi_data.loc[0,:].iloc[:2000], yokoi_data.loc[1,:].iloc[:200]])
validation = pd.concat([watara_data, yokoi_data.loc[0,:].iloc[2000:], yokoi_data.loc[1,:].iloc[200:]])

t_healthy = training.loc[0]
t_cancer = training.loc[1]
v_healthy = validation.loc[0]
v_cancer = validation.loc[1]

m_t_healthy = t_healthy.shape[0]
m_t_cancer = t_cancer.shape[0]
m_v_healthy = v_healthy.shape[0]
m_v_cancer = v_cancer.shape[0]

n_mirnas = t_healthy.shape[1] #gets number of columns
miRNA_names_list = list(t_healthy.columns)

print("healthy training data")
print(t_healthy.shape)
print(t_healthy.head)
print("cancer training data")
print(t_cancer.shape)
print(t_cancer.head)
print("healthy validation data")
print(v_healthy.shape)
print(v_healthy.head)
print("cancer validation data")
print(v_cancer.shape)
print(v_cancer.head)


healthy training data
(6965, 2565)
<bound method NDFrame.head of     MIMAT0000062  MIMAT0000063  MIMAT0000064  MIMAT0000065  MIMAT0000066  \
0          3.700         3.669        -2.681         3.499        -2.681   
0         -4.018        -4.018        -4.018        -4.018        -4.018   
0          1.514         0.869         0.464        -0.987        -4.325   
0         -4.417        -4.417        -4.417        -1.036        -1.897   
0         -5.541         0.598        -5.281        -3.185         0.986   
..           ...           ...           ...           ...           ...   
0         -3.515         1.626         6.450        -3.515         1.136   
0         -3.160        -3.160        -3.160        -3.160        -3.160   
0         -3.717         0.509        -3.717        -3.717         1.166   
0         -2.374        -2.374         1.426         2.128         3.553   
0         -5.741         1.007        -5.741        -5.741        -5.741   

    MIMAT0000067  MIMA

In [79]:
miRNA_names_list

['MIMAT0000062',
 'MIMAT0000063',
 'MIMAT0000064',
 'MIMAT0000065',
 'MIMAT0000066',
 'MIMAT0000067',
 'MIMAT0000068',
 'MIMAT0000069',
 'MIMAT0000070',
 'MIMAT0000071',
 'MIMAT0000072',
 'MIMAT0000073',
 'MIMAT0000074',
 'MIMAT0000075',
 'MIMAT0000076',
 'MIMAT0000077',
 'MIMAT0000078',
 'MIMAT0000079',
 'MIMAT0000080',
 'MIMAT0000081',
 'MIMAT0000082',
 'MIMAT0000083',
 'MIMAT0000084',
 'MIMAT0000085',
 'MIMAT0000086',
 'MIMAT0000087',
 'MIMAT0000088',
 'MIMAT0000089',
 'MIMAT0000090',
 'MIMAT0000091',
 'MIMAT0000092',
 'MIMAT0000093',
 'MIMAT0000094',
 'MIMAT0000095',
 'MIMAT0000096',
 'MIMAT0000097',
 'MIMAT0000098',
 'MIMAT0000099',
 'MIMAT0000100',
 'MIMAT0000101',
 'MIMAT0000102',
 'MIMAT0000103',
 'MIMAT0000104',
 'MIMAT0000222',
 'MIMAT0000226',
 'MIMAT0000227',
 'MIMAT0000228',
 'MIMAT0000231',
 'MIMAT0000232, MIMAT0004563',
 'MIMAT0000241',
 'MIMAT0000242',
 'MIMAT0000243',
 'MIMAT0000244',
 'MIMAT0000245',
 'MIMAT0000250',
 'MIMAT0000251',
 'MIMAT0000252',
 'MIMAT0000253',


In [4]:
import numpy as np
a = np.array([1,2,3,4])
b = np.array([2,3,4,5])
np.greater(a,b)

array([False, False, False, False])

In [101]:
W = t_healthy.values
print(W.shape)

(6965, 2565)


In [97]:
CONST_k = 0.95
significant_pairs = np.zeros((n_mirnas, n_mirnas))
for i in range(0,n_mirnas):
    W_current_miRNA = W[:, i:i+1] #shape (m_samples, 1)
    W_rest_miRNA = W[:, i+1:] #shape (m_samples, n_mirnas-i-1)
    W_comparisons = W_rest_miRNA < W_current_miRNA
    W_greater = np.sum(W_comparisons, axis=0) / m_t_healthy
    W_mask_greater_correlation = np.add((W_greater >= CONST_k).nonzero(), i) #-1
    W_mask_less_correlation = np.add((W_greater <= (1-CONST_k)).nonzero(), i) #-1
    significant_pairs[i][[W_mask_greater_correlation]] = 1
    significant_pairs[i][[W_mask_less_correlation]] = -1


In [103]:
a = 2427
b = 745
print(significant_pairs[2427,745])
testsum = 0.0
for i in range(0,m_t_healthy):
    if(W[i,a] > W[i,b]):
        testsum+=1
testsum/=m_t_healthy
print(testsum)

0.0
0.9750179468772434


In [9]:
testindicies1 = significant_pairs.nonzero()
print(len(testindicies1[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies1[0])):
    if(significant_pairs[testindicies1[0][i]][testindicies1[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": 1.0")
    else:
        no_greaterthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

1303398
number of Ea<Eb with a<b :413117
number of Ea>Eb with a<b:890281


In [23]:
X = t_cancer.values
X.shape

(200, 2565)

In [16]:
CONST_l = 0.95
cancer_pairs = np.zeros((n_mirnas, n_mirnas))
diagnostic_pairs = np.zeros((n_mirnas, n_mirnas))
for i in range(0,n_mirnas):
    X_current_miRNA = X[:, i:i+1] #shape (m_samples, 1)
    X_rest_miRNA = X[:, i+1:] #shape (m_samples, n_mirnas-i-1)
    X_comparisons = X_rest_miRNA < X_current_miRNA
    X_greater = np.sum(X_comparisons, axis=0) / m_t_cancer
    X_mask_greater_correlation = np.add((X_greater >= CONST_l).nonzero(), i) #-1
    X_mask_less_correlation = np.add((X_greater <= (1-CONST_l)).nonzero(), i) #-1
    cancer_pairs[i][[X_mask_greater_correlation]] = 1
    cancer_pairs[i][[X_mask_less_correlation]] = -1
    
    #diagnostic pairs are 
    # significant[mirna1][mirna2] == 1 and cancer[mirna1][mirna2] == -1
    # or significant[mirna][mirna2] == -1 and cancer[mirna1][mirna2] == 1
    #mirna1 is fixed at i
mask1 = np.logical_and(cancer_pairs==1,significant_pairs==-1)
mask2 = np.logical_and(cancer_pairs==-1,significant_pairs==1)
diagnostic_pairs = np.logical_or(mask1, mask2)



In [12]:
testindicies2 = cancer_pairs.nonzero()
print(len(testindicies2[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies2[0])):
    if(significant_pairs[testindicies2[0][i]][testindicies2[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies2[0][i])+", " + str(testindicies2[1][i]) +": 1.0")
    else:
        no_greaterthantest+=1
        #print(str(testindicies2[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

864722
number of Ea<Eb with a<b :276218
number of Ea>Eb with a<b:588504


In [42]:
testindicies3 = np.nonzero(diagnostic_pairs)
print(len(testindicies3[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies3[0])):
    if(significant_pairs[testindicies3[0][i]][testindicies3[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": 1.0")
    else: 
        no_greaterthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

3
number of Ea<Eb with a<b :2
number of Ea>Eb with a<b:1


In [88]:
diag_pairs = np.nonzero(diagnostic_pairs)

#for testing purposes, im going to use the og mirna pairs that the study uses. 
og_set1 = ['MIMAT0027686','MIMAT0027500','MIMAT0027500','MIMAT0030420',
           'MIMAT0027426','MIMAT0027468','MIMAT0022967','MIMAT0030420',
          'MIMAT0027500', 'MIMAT0004957','MIMAT0023714','MIMAT0027472',
          'MIMAT0027488']
og_set2 = ['MIMAT0005880','MIMAT0005929','MIMAT0005880','MIMAT0019957',
           'MIMAT0022947','MIMAT0022259','MIMAT0005929','MIMAT0000510',
           'MIMAT0000690','MIMAT0004952','MIMAT0019071','MIMAT0018085',
           'MIMAT0005880']
miRNA_names_list
#convert miR ids to indexes
og_set1_indexes = [miRNA_names_list.index(i) for i in og_set1]
og_set2_indexes = [miRNA_names_list.index(i) for i in og_set2]
diag_pairs = (np.array(og_set1_indexes), np.array(og_set2_indexes))
print(diag_pairs)

diagnostic_pairs_flat = np.concatenate((diag_pairs[0],diag_pairs[1]))
diag_dict = np.zeros((len(diag_pairs[0]), 2), dtype=int)
for i in range(len(diag_pairs[0])):
    _first_elem_index = np.where(diagnostic_pairs_flat==diag_pairs[0][i])[0][0] #weird ahh 3d array
    _second_elem_index = np.where(diagnostic_pairs_flat ==diag_pairs[1][i])[0][0]
    diag_dict[i] = np.array([_first_elem_index, _second_elem_index])
#diagnostic_pairs is binary

print(diagnostic_pairs_flat)
print(diag_dict)

#nxm array, this time n is 2 * # of diagnostic pairs
#m is # of samples
t_diagnostic_arr = np.concatenate((W[:,diagnostic_pairs_flat], X[:,diagnostic_pairs_flat]), axis=0)


(array([2427, 2241, 2241, 2492, 2167, 2209, 1912, 2492, 2241,  673, 1942,
       2213, 2229]), array([ 745,  791,  745, 1676, 1908, 1773,  791,  135,  148,  668, 1347,
       1123,  745]))
[2427 2241 2241 2492 2167 2209 1912 2492 2241  673 1942 2213 2229  745
  791  745 1676 1908 1773  791  135  148  668 1347 1123  745]
[[ 0 13]
 [ 1 14]
 [ 1 13]
 [ 3 16]
 [ 4 17]
 [ 5 18]
 [ 6 14]
 [ 3 20]
 [ 1 21]
 [ 9 22]
 [10 23]
 [11 24]
 [12 13]]


In [92]:

# see the performance of the diagnostic pairs.
# go thro each sample
# check each pair, and find majority. then diagnose it bruh.
t_correct_diagnosis_arr = np.zeros(m_t_cancer + m_t_healthy)
for i in range(m_t_healthy+m_t_cancer):
    _count = 0
    if(i == 7000):
        7000
    for j in range(len(diag_pairs[0])):
        _reo = t_diagnostic_arr[i,diag_dict[j,0]] > t_diagnostic_arr[i,diag_dict[j,1]]
        # _cancer_norm = cancer_pairs[diag_pairs[0][j],diag_pairs[1][j]] 
        # _diagnosis = False
        # if(_reo == False and _cancer_norm == -1) or (_reo and _cancer_norm): #reo is either 0 if smaller, 1 if larger. cancer norm is -1 if smaller, 1 if larger
        #     _diagnosis = True 
        _healthy_norm = significant_pairs[diag_pairs[0][j],diag_pairs[1][j]] 
        _diagnosis = True
        if(_reo == False and _healthy_norm == -1) or (_reo and _healthy_norm): #reo is either 0 if smaller, 1 if larger. cancer norm is -1 if smaller, 1 if larger
            _diagnosis = False
        if((i >= m_t_healthy and _diagnosis) or (i < m_t_healthy and not _diagnosis)):
            _count+=1
    if(_count >int(len(diag_pairs[0])/2)):
        t_correct_diagnosis_arr[i]=1

In [96]:
print(diag_pairs)
a,b = diag_pairs[0][0], diag_pairs[0][1]
print(significant_pairs[a][b])
print(cancer_pairs[a][b])

(array([2427, 2241, 2241, 2492, 2167, 2209, 1912, 2492, 2241,  673, 1942,
       2213, 2229]), array([ 745,  791,  745, 1676, 1908, 1773,  791,  135,  148,  668, 1347,
       1123,  745]))
0.0
0.0


In [93]:
print(np.sum(t_correct_diagnosis_arr))
print(np.shape(t_correct_diagnosis_arr))

200.0
(7165,)


In [ ]:
#make a (m_t_healthy+ m_t_cancer),n,n array with the indexes as 0/1 to indicate healthy/cancer sample
#for a sample mk (could be from healthy or cancer) you have a pair (a,b) and thus arr[a,b,mk] details 
#if Ea > Eb in that sample
healthy_comp_arr = np.zeros((m_t_healthy, n_mirnas, n_mirnas), dtype = bool)
for i in range(0,m_t_healthy):
    # you need to use np.greater(arr1, arr2)
    #where arr1 has the mirna expression the same in rows
    #and arr2 has the mirna expression the same in columns
    #more in notebook, 10/21/25 heading
    healthy_arr1 = np.repeat(W[i].reshape(1,-1), n_mirnas, axis=0)
    healthy_arr2 = np.repeat(W[i].reshape(-1,1), n_mirnas, axis=1)
    healthy_comp_arr[i] = np.greater(healthy_arr1, healthy_arr2)

NameError: name 'np' is not defined

In [ ]:
#very similar to previous
cancer_comp_arr = np.zeros((m_t_cancer, n_mirnas, n_mirnas), dtype=bool)
for i in range (0,m_t_cancer):
    cancer_arr1 = np.repeat(X[i].reshape(1,-1), n_mirnas, axis=0)
    cancer_arr2 = np.repeat(X[i].reshape(-1,1), n_mirnas, axis=1)
    cancer_comp_arr[i] = np.greater(cancer_arr1, cancer_arr2)
    